# 02 - Calas Data Preparation - All Years

Loads raw anchoveta fishing records (calas) from 2015-2024, aggregates them into weekly spatial grids, and computes season-level statistics.

**Steps:**
1. Load CSV catch records for each year and season from the IHMA data folder
2. Create weekly spatial grids (lat/lon bins) of total catch tonnage
3. Compute statistics by season and company (dates, duration, volume)
4. Save aggregated catch data, gridded weekly summaries, and statistics to outputs

In [1]:
from pyproj import datadir
datadir.set_data_dir("/home/jupyter-daniela/.conda/envs/peru_environment/share/proj")

import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
from pathlib import Path
import numpy as np
import re
import warnings
warnings.filterwarnings("ignore")
import json

In [2]:
def read_csv_auto(path):
    for enc in ('utf-8-sig', 'latin1'):
        try:
            return pd.read_csv(path, low_memory=False, encoding=enc)
        except UnicodeDecodeError:
            continue
    raise ValueError(f"Cannot decode {path}")

def load_year(base_path, anio):
    carpeta = Path(base_path) / "ihma_data" / str(anio)
    archivos = [
        a for a in carpeta.glob("*.csv")
        if re.search(r'Calas', a.name, re.IGNORECASE)
        and (re.search(r'Primera temporada', a.name, re.IGNORECASE) or re.search(r'Segunda temporada', a.name, re.IGNORECASE))
        and re.search(r'anchoveta', a.name, re.IGNORECASE)
    ]
    if not archivos:
        raise FileNotFoundError(f"No files found in {carpeta}")

    print(f"  {anio}: {[a.name for a in archivos]}")
    df = pd.concat([read_csv_auto(a) for a in archivos], ignore_index=True)
    df.columns = df.columns.str.strip()
    df = df.loc[:, ~df.columns.duplicated()]
    df['fecha_cala'] = pd.to_datetime(df['fecha_cala'], format='%d/%m/%Y')
    df['declarado_tm'] = pd.to_numeric(df['declarado_tm'], errors='coerce')
    if 'porcentaje_juvenil' in df.columns:
        df['porcentaje_juvenil'] = pd.to_numeric(df['porcentaje_juvenil'], errors='coerce')
    df['semana'] = df['fecha_cala'].dt.isocalendar().week
    print(f"    -> {len(df):,} rows")
    return df

In [3]:
base_path = "/home/jupyter-daniela/suyana/peru_production/inputs/"
output_path = Path("/home/jupyter-daniela/suyana/peru_production/outputs/")
years = range(2015, 2026)

dfs = []
for anio in years:
    try:
        dfs.append(load_year(base_path, anio))
    except FileNotFoundError:
        print(f"  Skipping {anio}: no files found")

  2015: ['Segunda temporada de anchoveta – Calas y biometría.csv']
    -> 7,277 rows
  2016: ['Segunda temporada de anchoveta – Calas y biometría.csv', 'Primera temporada de anchoveta – Calas y biometría.csv']


    -> 23,003 rows
  2017: ['Segunda temporada de anchoveta – Calas y biometría.csv', 'Primera temporada de anchoveta – Calas y biometría.csv']


    -> 20,031 rows
  2018: ['Segunda temporada de anchoveta – Calas y biometría.csv', 'Primera temporada de anchoveta – Calas y biometría.csv']


    -> 36,304 rows
  2019: ['Segunda temporada de anchoveta – Calas y biometría.csv', 'Primera temporada de anchoveta – Calas y biometría.csv']


    -> 28,407 rows
  2020: ['Segunda temporada de anchoveta – Calas y biometría.csv', 'Primera temporada de anchoveta – Calas y biometría.csv']


...

    -> 32,661 rows
  2022: ['Primera temporada de pesca de anchoveta – Calas y biometría.csv', 'Segunda temporada de pesca de anchoveta – Calas y biometría.csv']


    -> 38,158 rows
  2023: ['Segunda temporada de anchoveta – Calas y biometría.csv', 'Primera temporada de anchoveta – Calas y biometría_1.csv']


    -> 32,216 rows
  2024: ['Segunda temporada de anchoveta – Calas y biometría.csv', 'Primera temporada de anchoveta – Calas y biometría.csv']


    -> 65,863 rows
  2025: ['Segunda temporada de anchoveta – Calas y biometría.csv', 'Primera temporada de anchoveta – Calas y biometría.csv']


    -> 41,203 rows


In [4]:
df_todos = pd.concat(dfs, ignore_index=True)
print(f"Total rows: {len(df_todos):,}")

df_todos.to_csv(output_path / "calas_all_data.csv", index=False)
print(f"Saved -> {output_path / 'calas_all_data.csv'}")

Total rows: 357,684


Saved -> /home/jupyter-daniela/suyana/peru_production/outputs/calas_all_data.csv
